# Recovery Channel Proportion Prediction — Softmax Architecture

**Stage-1** (pretrained two-stage recovery model): predicts `p_recovered_hat` = P(recovered > 0) × E(rate | recovered > 0)

**Stage-2** (this notebook): for each of the 9 channels, train a regressor on the
**recovered subset** (rows where `prob_recovered > 0`) predicting the channel share
`prob_<channel> / prob_recovered`. At inference, apply softmax across the 9 channel
logits per row so they sum to 1, then multiply by `p_recovered_hat`:

```
pred_channel_X = p_recovered_hat * softmax(logit_1..9)_X
```

This guarantees the 9 channel predictions sum to `p_recovered_hat` per row
(internally consistent with Stage-1).

In [ ]:
# Install dependencies into the running kernel.
# polars must NOT use --no-deps: recent polars versions ship their Rust binary as
# a separate runtime package, and --no-deps skips it (leaves Python wrapper without
# its backend). shap/seaborn deps are already in the kernel so --no-deps is fine.
# xgboost pinned to a prebuilt wheel since xgboost>=3.2 source build needs CMake>=3.18.
%pip install --quiet --force-reinstall --only-binary=:all: polars
%pip install --quiet --only-binary=:all: "xgboost==2.1.4"
%pip install --quiet --no-deps shap seaborn psutil

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math
import pandas as pd
import polars as pl
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
)
from xgboost import XGBClassifier, XGBRegressor
import time
import joblib
import boto3
import tempfile
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(15)

In [ ]:
def save_model_to_s3(model, bucket: str, key: str) -> None:
    """Serialize a model with joblib and upload to S3."""
    s3 = boto3.client("s3")
    with tempfile.NamedTemporaryFile(suffix=".joblib") as tmp:
        joblib.dump(model, tmp.name)
        s3.upload_file(tmp.name, bucket, key)
    logger.info(f"Saved model to s3://{bucket}/{key}")


def load_model_from_s3(bucket: str, key: str):
    """Download a model from S3 and deserialize with joblib."""
    s3 = boto3.client("s3")
    with tempfile.NamedTemporaryFile(suffix=".joblib") as tmp:
        s3.download_file(bucket, key, tmp.name)
        model = joblib.load(tmp.name)
    logger.info(f"Loaded model from s3://{bucket}/{key}")
    return model

In [ ]:
# Lazy scan — opens parquet metadata without loading data into memory.
# An eager read of the full ~8.7M x 820 frame would exceed available memory
# once it gets filtered into train and test. Lazy scanning lets polars push
# the row filter and column projection down to the parquet reader, so only
# the rows and columns actually needed are materialized.
S3_PARQUET_URI = "s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_full_features.parquet"

df_lazy = pl.scan_parquet(S3_PARQUET_URI)
all_cols = df_lazy.collect_schema().names()
print(f"Parquet schema: {len(all_cols)} columns total (no data loaded yet)")

In [ ]:
# Discovery: list per-channel temporal columns already in the parquet.
# Uses the lazy schema (no data loaded) to enumerate columns by name.
_channels = [
    "remove_return", "bintool_donations", "donations",
    "warehouse_deals_and_gr", "liquidations", "return_to_vendor",
    "bintool_theft", "remove_liquidate", "bintool_remove_liquidate",
]
print("=" * 60)
print("Per-channel temporal columns discovery")
print("=" * 60)
for short in _channels:
    gl_lag    = sorted(c for c in all_cols if c.startswith(f"prob_{short}_lag_"))
    gl_roll   = sorted(c for c in all_cols if c.startswith(f"prob_{short}_rolling_"))
    gl_ewma   = sorted(c for c in all_cols if c.startswith(f"prob_{short}_ewma_"))
    site_lag  = sorted(c for c in all_cols if c.startswith(f"site_prob_{short}_week_lag_"))
    site_roll = sorted(c for c in all_cols if c.startswith(f"site_prob_{short}_week_rolling_"))
    site_ewma = sorted(c for c in all_cols if c.startswith(f"site_prob_{short}_week_ewma_"))
    print(f"\n--- {short} ---")
    print(f"  GL-site lag     ({len(gl_lag)}): {gl_lag}")
    print(f"  GL-site rolling ({len(gl_roll)}): {gl_roll}")
    print(f"  GL-site ewma    ({len(gl_ewma)}): {gl_ewma}")
    print(f"  site lag        ({len(site_lag)}): {site_lag}")
    print(f"  site rolling    ({len(site_roll)}): {site_roll}")
    print(f"  site ewma       ({len(site_ewma)}): {site_ewma}")

In [ ]:
# Lazy filter — no materialization yet, just defines the train/test plans.
# Actual collect happens in the column-shrink cell below, after we've defined
# feature_cols + needed_cols, so we can also push the column projection down.
df_train_lazy = df_lazy.filter(pl.col("year") < 2025)
df_test_lazy  = df_lazy.filter(pl.col("year") == 2025)
print("Train/test plans built (lazy — nothing loaded yet).")

In [ ]:
# -- Identity columns --
identity_cols = ["hashed_fc", "gl_product_group", "year", "week"]

# -- GL composition --
gl_composition_cols = [
    "share_food", "share_non_food", "share_pet_food",
    "share_RETAIL", "share_FBA", "share_hazmat",
]

# -- GL volume --
gl_volume_cols = [
    "units_total", "cogs_total", "weight_total",
    "avg_cogs_per_unit", "avg_weight_per_unit", "cogs_per_unit_weight",
]

# -- GL at-site share --
gl_at_site_cols = [
    "site_units_share_week", "site_weight_share_week",
]

# -- Site context --
site_context_cols = [
    "site_units_total_week", "site_weight_total_week",
    "site_type", "site_category", "country", "country_state",
]

# -- Temporal site context --
temporal_site_context_cols = [
    f"site_units_total_week_lag_{w}w" for w in [1, 4, 12, 13, 52]
] + [
    f"site_weight_total_week_lag_{w}w" for w in [1, 4, 12, 13, 52]
] + [
    f"site_prob_recovered_week_lag_{w}w" for w in [1, 4, 12, 13, 52]
] + [
    f"site_prob_recovered_week_rolling_{w}w" for w in [4, 12, 26, 52]
]

# -- Calendar --
calendar_cols = ["month", "week"]

In [ ]:
# -- Temporal composition --
temporal_composition_cols = (
    [
        f"share_{c}_lag_{w}w"
        for c in ["RETAIL", "FBA", "hazmat", "food", "non_food", "pet_food"]
        for w in [1, 4, 12, 13, 52]
        if not (c == "FBA" and w == 1)  # share_FBA_lag_1w is missing
    ]
    + [
        f"share_{c}_rolling_{w}w"
        for c in ["food", "non_food", "pet_food", "RETAIL", "FBA", "hazmat"]
        for w in [4, 12, 26, 52]
    ]
    + [
        f"share_{c}_ewma_{a}"
        for c in ["RETAIL", "FBA", "hazmat", "food", "non_food", "pet_food"]
        for a in ["5a", "1a"]
    ]
)

# -- Temporal volume --
temporal_volume_cols = [
    f"{v}_lag_{w}w"
    for v in ["units_total", "cogs_total", "weight_total"]
    for w in [1, 4, 12, 13, 52]
] + [
    f"{v}_rolling_{w}w"
    for v in ["units_total", "cogs_total", "weight_total"]
    for w in [4, 12, 26, 52]
] + [
    f"{v}_ewma_{a}"
    for v in ["units_total", "cogs_total", "weight_total"]
    for a in ["5a", "1a"]
]

# -- Temporal probability (recovered) --
temporal_probability_cols = [
    f"prob_recovered_lag_{w}w" for w in [1, 4, 12, 13, 52]
] + [
    f"prob_recovered_rolling_{w}w" for w in [26, 52, 4, 12]
] + [
    f"prob_recovered_ewma_{a}" for a in ["5a", "1a"]
]

# -- Temporal per-channel probability (GL-site level) --
# Per-channel lag/rolling/EWMA features computed at the (site, GL) granularity.
_CH_SHORT = [
    "remove_return", "bintool_donations", "donations",
    "warehouse_deals_and_gr", "liquidations", "return_to_vendor",
    "bintool_theft", "remove_liquidate", "bintool_remove_liquidate",
]
temporal_per_channel_gl_cols = [
    f"prob_{ch}_lag_{w}w"
    for ch in _CH_SHORT
    for w in [1, 4, 12, 13, 52]
] + [
    f"prob_{ch}_rolling_{w}w"
    for ch in _CH_SHORT
    for w in [4, 12, 26, 52]
] + [
    f"prob_{ch}_ewma_{a}"
    for ch in _CH_SHORT
    for a in ["5a", "1a"]
]

# -- Temporal per-channel probability (site level) --
temporal_per_channel_site_cols = [
    f"site_prob_{ch}_week_lag_{w}w"
    for ch in _CH_SHORT
    for w in [1, 4, 12, 13, 52]
] + [
    f"site_prob_{ch}_week_rolling_{w}w"
    for ch in _CH_SHORT
    for w in [4, 12, 26, 52]
] + [
    f"site_prob_{ch}_week_ewma_{a}"
    for ch in _CH_SHORT
    for a in ["5a", "1a"]
]

# Filter the per-channel feature lists against the parquet schema so
# anything missing upstream is silently dropped instead of crashing later.
_available = set(all_cols)
temporal_per_channel_gl_cols   = [c for c in temporal_per_channel_gl_cols   if c in _available]
temporal_per_channel_site_cols = [c for c in temporal_per_channel_site_cols if c in _available]

# -- All features --
feature_cols = (
    gl_composition_cols
    + gl_volume_cols
    + gl_at_site_cols
    + site_context_cols
    + temporal_site_context_cols
    + calendar_cols
    + temporal_composition_cols
    + temporal_volume_cols
    + temporal_probability_cols
    + temporal_per_channel_gl_cols
    + temporal_per_channel_site_cols
)
print(f"Total features: {len(feature_cols)}")
print(f"  per-channel GL  : {len(temporal_per_channel_gl_cols)}")
print(f"  per-channel site: {len(temporal_per_channel_site_cols)}")

In [ ]:
EPS = 1e-7


def logit(p: np.ndarray) -> np.ndarray:
    """Clip-safe logit transform."""
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def sigmoid(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""
    return 1 / (1 + np.exp(-x))


# month and week are kept numeric. XGBoost handles them as-is, and casting
# float-derived columns to Categorical caused stability issues during training.
CAT_COLS = ["site_type", "site_category", "country", "country_state"]


def cast_categoricals(df: pl.DataFrame) -> pl.DataFrame:
    """Cast known categorical columns to Categorical dtype."""
    exprs = []
    for c in CAT_COLS:
        if c in df.columns and df.schema[c] != pl.Categorical:
            exprs.append(pl.col(c).cast(pl.Utf8).cast(pl.Categorical))
    return df.with_columns(exprs) if exprs else df


def prob_mae(y_hat: np.ndarray, dtrain) -> tuple[str, float]:
    """Custom XGBoost eval metric: MAE in probability space."""
    y_true = dtrain.get_label()
    pred_prob = sigmoid(y_hat)
    true_prob = sigmoid(y_true)
    mae = float(np.mean(np.abs(pred_prob - true_prob)))
    return "prob_mae", mae


def _rss_gb() -> float:
    """Return current process RSS in GB. Best-effort; 0 if psutil unavailable."""
    try:
        import psutil, os
        return psutil.Process(os.getpid()).memory_info().rss / (1024**3)
    except Exception:
        return 0.0


def polars_to_pandas_safe(df: pl.DataFrame) -> pd.DataFrame:
    """Convert a polars DataFrame to pandas column-by-column to avoid arrow
    chunked-array conversion failures and reduce peak memory. Logs progress
    every 25 columns with current process RSS."""
    n_cols = len(df.columns)
    data = {}
    for j, col in enumerate(df.columns):
        s = df[col]
        if s.dtype == pl.Categorical:
            data[col] = pd.Categorical(s.to_numpy())
        else:
            data[col] = s.to_numpy()
        if (j + 1) % 25 == 0 or (j + 1) == n_cols:
            print(f"      [convert] {j+1}/{n_cols} cols done  RSS={_rss_gb():.1f} GB", flush=True)
    out = pd.DataFrame(data)
    print(f"      [convert] pd.DataFrame assembled  RSS={_rss_gb():.1f} GB", flush=True)
    return out

In [ ]:
RECOVERY_CHANNELS = [
    "prob_remove_return",
    "prob_bintool_donations",
    "prob_donations",
    "prob_warehouse_deals_and_gr",
    "prob_liquidations",
    "prob_return_to_vendor",
    "prob_bintool_theft",
    "prob_remove_liquidate",
    "prob_bintool_remove_liquidate",
]
print(f"Channels: {len(RECOVERY_CHANNELS)}")

In [ ]:
# Collect train/test from S3 with predicate + projection pushdown.
# Memory usage = filtered_rows x needed_cols only. No full-width frame ever exists.
import gc

needed_cols = (
    set(identity_cols) | set(feature_cols) | set(RECOVERY_CHANNELS)
    | {"year", "month", "week", "prob_recovered"}
)
# Keep only columns that actually exist in parquet
_avail = set(all_cols)
needed_cols = sorted(needed_cols & _avail)
print(f"Selecting {len(needed_cols)} columns from parquet (out of {len(all_cols)})")

print("Collecting df_train...", flush=True)
df_train = df_train_lazy.select(needed_cols).collect()
print(f"  df_train: {df_train.shape}  RSS={_rss_gb():.1f} GB")

print("Collecting df_test...", flush=True)
df_test = df_test_lazy.select(needed_cols).collect()
print(f"  df_test:  {df_test.shape}  RSS={_rss_gb():.1f} GB")

# Drop the lazy plans (small, but no longer needed)
del df_train_lazy, df_test_lazy, df_lazy
gc.collect()
print(f"After cleanup RSS: {_rss_gb():.1f} GB")

# Stage-2: Per-Channel Share Regressor (softmax)

In [ ]:
# Diagnostic: enumerate model artifacts in S3 to confirm the keys used for
# stage1_clf / stage1_reg below. Run this if model keys ever change upstream.
import boto3

s3 = boto3.client("s3")

for prefix in ["model", "models"]:
    print("=== Prefix:", prefix, "===")
    resp = s3.list_objects_v2(Bucket="msds-26.2-data", Prefix=prefix)
    contents = resp.get("Contents", [])
    if not contents:
        print("  (empty)")
    for obj in contents:
        print(" ", obj["Size"], "bytes  ", obj["Key"])

In [ ]:
# ---- Load Stage-1 overall recovery models from S3 ----
# Stage-1 is loaded from the tuned variants in S3, which were trained on the
# full row set. Update STAGE1_*_KEY below if the upstream artifacts move.
STAGE1_BUCKET = "msds-26.2-data"
STAGE1_CLF_KEY = "model/tuned_xgboost_classification_model.joblib"
STAGE1_REG_KEY = "model/tuned_xgboost_regression_model.joblib"

stage1_clf = load_model_from_s3(STAGE1_BUCKET, STAGE1_CLF_KEY)
stage1_reg = load_model_from_s3(STAGE1_BUCKET, STAGE1_REG_KEY)

# Pull the exact feature names + order each Stage-1 model was trained on.
stage1_clf_features = list(stage1_clf.get_booster().feature_names)
stage1_reg_features = list(stage1_reg.get_booster().feature_names)

# The Stage-1 regressor may or may not carry a .site_gl_baseline_ attribute
# (a polars DataFrame keyed on hashed_fc + gl_product_group). Handle both cases.
stage1_site_gl_baseline = getattr(stage1_reg, "site_gl_baseline_", None)
if stage1_site_gl_baseline is None:
    print("NOTE: stage1_reg has no .site_gl_baseline_ attribute — will compute one from training data below.")
else:
    print(f"stage1_reg.site_gl_baseline_ shape: {stage1_site_gl_baseline.shape}")

print(f"Stage-1 clf expects {len(stage1_clf_features)} features")
print(f"Stage-1 reg expects {len(stage1_reg_features)} features")

# Sanity: which Stage-1 features are missing from df_train?
_baseline_cols = ("site_gl_mean_rate", "site_gl_std_rate", "site_gl_n_nonzero_weeks",
                  "baseline_mean", "baseline_std", "baseline_count")
missing_clf = [c for c in stage1_clf_features if c not in df_train.columns]
missing_reg = [c for c in stage1_reg_features
               if c not in df_train.columns and c not in _baseline_cols]
if missing_clf:
    print("WARN missing clf features in df_train:", missing_clf[:10], "...")
if missing_reg:
    print("WARN missing reg features in df_train:", missing_reg[:10], "...")
print("Sample stage1_reg feature names:", stage1_reg_features[:5], "...", stage1_reg_features[-5:])

In [ ]:
# ---- Compute Stage-1 p_recovered_hat on full train and test ----
print("Casting + converting features...", flush=True)
df_train_cast = cast_categoricals(df_train)
df_test_cast  = cast_categoricals(df_test)

X_train_clf = polars_to_pandas_safe(df_train_cast.select(stage1_clf_features))
X_test_clf  = polars_to_pandas_safe(df_test_cast.select(stage1_clf_features))

print("Stage-1 classifier: P(recovered > 0)...", flush=True)
p_nonzero_train = stage1_clf.predict_proba(X_train_clf)[:, 1]
p_nonzero_test  = stage1_clf.predict_proba(X_test_clf)[:, 1]

# Stage-1 regressor needs site-GL baseline joined in.
print("Stage-1 regressor: joining baseline + predicting...", flush=True)
df_train_ext = df_train_cast.join(stage1_site_gl_baseline, on=["hashed_fc", "gl_product_group"], how="left")
df_test_ext  = df_test_cast.join(stage1_site_gl_baseline,  on=["hashed_fc", "gl_product_group"], how="left")

X_train_reg1 = polars_to_pandas_safe(df_train_ext.select(stage1_reg_features))
X_test_reg1  = polars_to_pandas_safe(df_test_ext.select(stage1_reg_features))

e_rate_train = np.clip(sigmoid(stage1_reg.predict(X_train_reg1)), 0.0, 1.0)
e_rate_test  = np.clip(sigmoid(stage1_reg.predict(X_test_reg1)),  0.0, 1.0)

p_recovered_hat_train = p_nonzero_train * e_rate_train
p_recovered_hat_test  = p_nonzero_test  * e_rate_test

print(f"p_recovered_hat_train: mean={p_recovered_hat_train.mean():.4f}  (actual: {df_train['prob_recovered'].mean():.4f})")
print(f"p_recovered_hat_test:  mean={p_recovered_hat_test.mean():.4f}  (actual: {df_test['prob_recovered'].mean():.4f})")

import gc
del X_train_clf, X_test_clf, X_train_reg1, X_test_reg1, df_train_ext, df_test_ext
gc.collect()

In [ ]:
# ---- Per-channel share regressor ----
# Target = prob_<channel> / prob_recovered (the share of recovery going to this channel).
# Trained on the recovered subset only (rows with prob_recovered > 0).

def train_channel_share_regressor(df_train_cast, df_test_cast, feature_cols, target_col):
    df_tr    = df_train_cast.filter(pl.col("prob_recovered") > 0)
    df_te_nz = df_test_cast.filter(pl.col("prob_recovered") > 0)

    print(f"  [share] recovered-subset rows  train={len(df_tr):,}  test_nz={len(df_te_nz):,}", flush=True)

    df_tr    = df_tr.with_columns((pl.col(target_col) / pl.col("prob_recovered")).alias("_share"))
    df_te_nz = df_te_nz.with_columns((pl.col(target_col) / pl.col("prob_recovered")).alias("_share"))

    site_gl_baseline = (
        df_tr.group_by(["hashed_fc", "gl_product_group"])
        .agg([
            pl.col("_share").mean().alias("baseline_share_mean"),
            pl.col("_share").std().alias("baseline_share_std"),
            pl.col("_share").count().alias("baseline_share_count"),
        ])
    )
    df_tr    = df_tr.join(site_gl_baseline, on=["hashed_fc", "gl_product_group"], how="left")
    df_te_nz = df_te_nz.join(site_gl_baseline, on=["hashed_fc", "gl_product_group"], how="left")

    aug_features = feature_cols + ["baseline_share_mean", "baseline_share_std", "baseline_share_count"]
    aug_features = [c for c in aug_features if c in df_tr.columns]

    X_train = polars_to_pandas_safe(df_tr.select(aug_features))
    X_te_nz = polars_to_pandas_safe(df_te_nz.select(aug_features))
    y_train_raw = df_tr["_share"].to_numpy()
    y_te_nz_raw = df_te_nz["_share"].to_numpy()
    y_train_lg  = logit(np.clip(y_train_raw, EPS, 1 - EPS))
    y_te_nz_lg  = logit(np.clip(y_te_nz_raw, EPS, 1 - EPS))

    model = XGBRegressor(
        n_jobs=-1,
        n_estimators=500,
        max_depth=5,
        learning_rate=0.15,
        subsample=0.7,
        colsample_bytree=0.7,
        tree_method="hist",
        enable_categorical=True,
        early_stopping_rounds=30,
        random_state=42,
        verbosity=0,
        eval_metric="mae",
    )
    model.fit(
        X_train, y_train_lg,
        eval_set=[(X_te_nz, y_te_nz_lg)],
        verbose=50,
    )

    pred_nz = np.clip(sigmoid(model.predict(X_te_nz)), 0.0, 1.0)
    mae  = mean_absolute_error(y_te_nz_raw, pred_nz)
    rmse = math.sqrt(mean_squared_error(y_te_nz_raw, pred_nz))
    r2   = r2_score(y_te_nz_raw, pred_nz)
    print(f"  [share]  best_iter={model.best_iteration}  recovered-subset MAE={mae:.4f}  R2={r2:.4f}", flush=True)

    return model, aug_features, site_gl_baseline, {"mae": mae, "rmse": rmse, "r2": r2}

In [ ]:
# ---- Phase 1: train all 9 share regressors and collect full-test logit predictions ----
import gc

share_models = {}
logit_test = np.zeros((len(df_test_cast), len(RECOVERY_CHANNELS)), dtype=np.float32)

for i, ch in enumerate(RECOVERY_CHANNELS):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(RECOVERY_CHANNELS)}] Channel: {ch}")
    print("="*60, flush=True)

    model, aug_features, site_gl_baseline, reg_metrics = train_channel_share_regressor(
        df_train_cast, df_test_cast, feature_cols, ch
    )

    df_te_ext = df_test_cast.join(site_gl_baseline, on=["hashed_fc", "gl_product_group"], how="left")
    X_te_full = polars_to_pandas_safe(df_te_ext.select(aug_features))
    logit_test[:, i] = model.predict(X_te_full)

    share_models[ch] = {
        "model": model,
        "aug_features": aug_features,
        "site_gl_baseline": site_gl_baseline,
        "reg_metrics": reg_metrics,
    }

    del X_te_full, df_te_ext
    gc.collect()

print("\nAll share regressors trained. logit_test shape:", logit_test.shape)

In [ ]:
# ---- Phase 2: softmax across channels, combine with p_recovered_hat ----
# Softmax over 9 channel logits per row -> valid distribution summing to 1.
# pred_channel_X = p_recovered_hat * softmax_share_X
logit_shift    = logit_test - logit_test.max(axis=1, keepdims=True)
exp_logit      = np.exp(logit_shift)
softmax_shares = exp_logit / exp_logit.sum(axis=1, keepdims=True)

print(f"softmax_shares shape: {softmax_shares.shape}")
row_sums = softmax_shares.sum(axis=1)
print(f"row sum sanity (should be ~1.0): min={row_sums.min():.4f}  max={row_sums.max():.4f}")

combined_preds = softmax_shares * p_recovered_hat_test[:, None]

share_results = {}
for i, ch in enumerate(RECOVERY_CHANNELS):
    y_true = df_test_cast[ch].to_numpy()
    pred   = combined_preds[:, i]
    mae    = mean_absolute_error(y_true, pred)
    rmse   = math.sqrt(mean_squared_error(y_true, pred))
    r2     = r2_score(y_true, pred)
    share_results[ch] = {
        **share_models[ch],
        "combined_metrics": {
            "mae": mae, "rmse": rmse, "r2": r2,
            "actual_mean":    float(y_true.mean()),
            "predicted_mean": float(pred.mean()),
        },
        "combined_pred":  pred,
        "y_true":         y_true,
        "softmax_share": softmax_shares[:, i],
    }
    print(f"  {ch:35s}  actual={y_true.mean():.5f}  pred={pred.mean():.5f}  ratio={pred.mean()/max(y_true.mean(),1e-9):.2f}x  MAE={mae:.5f}  R2={r2:.4f}")

In [ ]:
# ---- Summary table ----
rows = []
for ch, res in share_results.items():
    cm = res["combined_metrics"]
    rows.append({
        "channel": ch,
        "actual_mean":    cm["actual_mean"],
        "predicted_mean": cm["predicted_mean"],
        "ratio":          cm["predicted_mean"] / max(cm["actual_mean"], 1e-9),
        "mae":            cm["mae"],
        "rmse":           cm["rmse"],
        "r2":             cm["r2"],
    })

summary_df = pd.DataFrame(rows).sort_values("actual_mean", ascending=False)
print(summary_df.to_string(index=False))

# Save Share Models to S3 (optional)

In [ ]:
SHARE_PREFIX = "models/recovery_channel_share_softmax"

for ch in RECOVERY_CHANNELS:
    print(f"Saving {ch}...")
    save_model_to_s3(
        share_models[ch]["model"],
        STAGE1_BUCKET,
        f"{SHARE_PREFIX}/{ch}_share.joblib",
    )

print("All share models saved.")